In [ ]:
# =============================================================================
# 01_preprocessing.ipynb  —  HPC version (hardcoded absolute paths)
# =============================================================================

import os, re, sys, json, traceback
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import rioxarray as rxr
import rasterio
from rasterio.enums import Resampling
from tqdm.auto import tqdm

# ---------------- Paths (HPC) — HARDCODED, do not derive from cwd ----------------
REPO_ROOT = Path('/scratch/lustre/users/fngari/John/SOIL-MOISTURE-PREDICTION')
DATA_IN   = REPO_ROOT / 'data' / 'obj1'
DATA_OUT  = REPO_ROOT / 'data' / 'processed'
LOG_DIR   = REPO_ROOT / 'notebooks' / 'logs'

# ---------------- Fail-fast checks ----------------
assert REPO_ROOT.is_absolute(), f"REPO_ROOT must be absolute, got {REPO_ROOT}"
assert REPO_ROOT.exists(),     f"REPO_ROOT missing: {REPO_ROOT}"
assert DATA_IN.exists(),       f"DATA_IN missing: {DATA_IN}"
assert 'notebooks/SOIL-MOISTURE-PREDICTION' not in str(DATA_OUT), \
    f"Path duplication detected: {DATA_OUT}"

DATA_OUT.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

# ---------------- Config ----------------
SEASONS   = ['JF', 'MAM', 'JJAS', 'OND']
YEARS     = list(range(1995, 2026))
NODATA    = -9999

CRS       = 'EPSG:21037'
RES       = 30
NOBS_MIN  = 2

DATASETS = {
    'NDVI':       ('NDVI',       None),
    'LST':        ('LST',        None),
    'NOBS_NDVI':  ('NOBS_NDVI',  None),
    'NOBS_LST':   ('NOBS_LST',   None),
    'SM_L1':      ('SM_L1',      Resampling.bilinear),
    'SM_L2':      ('SM_L2',      Resampling.bilinear),
    'SM_L3':      ('SM_L3',      Resampling.bilinear),
    'SM_L4':      ('SM_L4',      Resampling.bilinear),
    'T2M':        ('T2M',        Resampling.bilinear),
    'T2M_MAX':    ('T2M_MAX',    Resampling.bilinear),
    'T2M_MIN':    ('T2M_MIN',    Resampling.bilinear),
    'PRECIP':     ('PRECIP',     Resampling.bilinear),
    'FIRE_DAYS':  ('FIRE_DAYS',  Resampling.nearest),
}

# ---------------- Sanity print ----------------
print(f"REPO_ROOT : {REPO_ROOT}")
print(f"DATA_IN   : {DATA_IN}   exists={DATA_IN.exists()}")
print(f"DATA_OUT  : {DATA_OUT}  exists={DATA_OUT.exists()}")
print(f"LOG_DIR   : {LOG_DIR}   exists={LOG_DIR.exists()}")
print(f"DATASETS  : {len(DATASETS)} variables")

In [ ]:
# =============================================================================
# Pre-flight check. Fails early if inputs are missing, so you don't waste
# 30 minutes on a run that was never going to work.
# =============================================================================

tif_files = sorted(DATA_IN.glob('*.tif'))
print(f"Found {len(tif_files)} GeoTIFF files in {DATA_IN}")

if len(tif_files) == 0:
    raise FileNotFoundError(
        f"No .tif files in {DATA_IN}. "
        f"Confirm the GEE exports landed here. "
        f"Contents of directory: {list(DATA_IN.iterdir())[:10]}"
    )

# Quick file size distribution — tiny files (<1 KB) are suspicious
sizes_kb = np.array([f.stat().st_size / 1024 for f in tif_files])
print(f"File size (KB): min={sizes_kb.min():.1f}  "
      f"median={np.median(sizes_kb):.1f}  max={sizes_kb.max():.1f}")

# Parse filenames into a manifest
pattern = re.compile(
    r'^(?P<var>[A-Z0-9_]+)_(?P<season>JF|MAM|JJAS|OND)_(?P<year>\d{4})\.tif$'
)
records = []
unmatched = []
for f in tif_files:
    m = pattern.match(f.name)
    if m:
        records.append({
            'path':   f,
            'var':    m.group('var'),
            'season': m.group('season'),
            'year':   int(m.group('year')),
        })
    else:
        unmatched.append(f.name)

manifest = pd.DataFrame(records)
print(f"\nParsed {len(manifest)} files, {len(unmatched)} unmatched.")

if unmatched:
    print("Unmatched filenames (first 10):")
    for n in unmatched[:10]:
        print(f"  {n}")

print("\nFiles per variable:")
print(manifest.groupby('var').size().sort_values(ascending=False))

manifest.to_csv(LOG_DIR / 'manifest.csv', index=False)

In [ ]:
# =============================================================================
# Build master grid from a reference Landsat file.
# We do NOT trust the CRS tag on the GeoTIFF blindly — we verify it and
# reproject to EPSG:21037 if needed.
# =============================================================================

# Pick the reference: prefer NDVI_JF_2010, fall back to any NDVI
ref_candidates = manifest[
    (manifest['var'] == 'NDVI') &
    (manifest['season'] == 'JF') &
    (manifest['year'] == 2010)
]
if len(ref_candidates) == 0:
    ref_candidates = manifest[manifest['var'] == 'NDVI'].head(1)
if len(ref_candidates) == 0:
    raise RuntimeError("No NDVI files found — cannot build master grid.")

ref_path = ref_candidates.iloc[0]['path']
print(f"Reference file: {ref_path.name}")

ref = rxr.open_rasterio(ref_path, masked=True).squeeze()

# Report source CRS
print(f"Source CRS: {ref.rio.crs}")
print(f"Source shape: {ref.shape}")
print(f"Source resolution: {ref.rio.resolution()}")

# Force to EPSG:21037 if not already
if ref.rio.crs is None:
    print("⚠ No CRS tag — assuming EPSG:21037")
    ref = ref.rio.write_crs(CRS, inplace=False)
elif str(ref.rio.crs) != CRS:
    print(f"⚠ Reprojecting from {ref.rio.crs} to {CRS}")
    ref = ref.rio.reproject(CRS, resolution=RES, resampling=Resampling.nearest)

master_transform = ref.rio.transform()
master_shape     = ref.shape
master_x         = ref.x.values
master_y         = ref.y.values

print(f"\nMaster grid:")
print(f"  Shape:     {master_shape[0]} rows × {master_shape[1]} cols")
print(f"  CRS:       {ref.rio.crs}")
print(f"  Transform: {master_transform}")
print(f"  Res:       {ref.rio.resolution()}")

# Sanity check on grid size — expect thousands, not tens
if master_shape[0] < 100 or master_shape[1] < 100:
    raise RuntimeError(
        f"Master grid is suspiciously small: {master_shape}. "
        f"Check that the reference GeoTIFF covers the AOI."
    )

# Save master grid metadata for later notebooks
np.savez(
    LOG_DIR / 'master_grid.npz',
    transform=np.array(master_transform).reshape(3, 3) if hasattr(master_transform, '__len__') else np.array(master_transform),
    shape=np.array(master_shape),
    x=master_x,
    y=master_y,
    crs=str(ref.rio.crs),
    res=np.array(ref.rio.resolution()),
)
print(f"✓ Saved master grid metadata to {LOG_DIR / 'master_grid.npz'}")

In [ ]:
# =============================================================================
# Load a single GeoTIFF and align to master grid.
# Returns NaN-filled array if the file is corrupt.
# =============================================================================

load_errors = []

def load_aligned(path, resampling=None):
    """Align one GeoTIFF to master grid. Returns float32 2D array."""
    try:
        da = rxr.open_rasterio(path, masked=True).squeeze()

        if da.rio.crs is None:
            da = da.rio.write_crs(CRS, inplace=False)

        if da.shape != master_shape or str(da.rio.crs) != CRS:
            if resampling is None:
                resampling = Resampling.bilinear
            da = da.rio.reproject(
                CRS,
                shape=master_shape,
                transform=master_transform,
                resampling=resampling,
            )

        arr = da.values.astype('float32')
        arr[arr == NODATA] = np.nan
        arr[~np.isfinite(arr)] = np.nan
        da.close()
        return arr

    except Exception as e:
        load_errors.append((str(path), repr(e)))
        print(f"  ✗ {Path(path).name}: {e}")
        return np.full(master_shape, np.nan, dtype='float32')

In [ ]:
# =============================================================================
# Build (time, y, x) stack per variable. Missing files → NaN slabs.
#
# NetCDF cannot serialize a MultiIndex, so time is stored as a plain integer
# index, with 'year' and 'season' attached as non-dimension coordinates.
#
# Helper for downstream selection:
#   sel_season(da, 2010, 'JF')  →  2D slice for Jan–Feb 2010
# =============================================================================

def build_variable_stack(var_name, resampling=None):
    rows = manifest[manifest['var'] == var_name]

    full_index = pd.MultiIndex.from_product(
        [YEARS, SEASONS], names=['year', 'season']
    )

    data = np.full(
        (len(full_index),) + master_shape,
        np.nan, dtype='float32'
    )

    for _, r in tqdm(rows.iterrows(), total=len(rows), desc=var_name):
        idx = full_index.get_loc((r['year'], r['season']))
        data[idx] = load_aligned(r['path'], resampling=resampling)

    # Split MultiIndex into plain coords for NetCDF serialization
    years_arr   = np.array([y for y, s in full_index], dtype='int32')
    seasons_arr = np.array([s for y, s in full_index], dtype='U4')

    da = xr.DataArray(
        data,
        dims=('time', 'y', 'x'),
        coords={
            'time':   np.arange(len(full_index), dtype='int32'),
            'year':   ('time', years_arr),
            'season': ('time', seasons_arr),
            'y':      master_y,
            'x':      master_x,
        },
        name=var_name,
        attrs={
            'crs':    CRS,
            'res':    RES,
            'nodata': 'NaN',
            'source': 'GEE seasonal export',
        },
    )
    da = da.rio.write_crs(CRS, inplace=False)
    da = da.rio.write_transform(master_transform, inplace=False)
    return da


# Convenience helper for downstream use
def sel_season(da, year, season):
    """Return 2D slice for a given (year, season)."""
    mask = (da.year.values == year) & (da.season.values == season)
    return da.isel(time=mask).squeeze()

In [ ]:
# =============================================================================
# Build and save each variable's stack, logging success/failure per variable.
# =============================================================================

run_log = []

for var, (_, resampling) in DATASETS.items():
    if var not in manifest['var'].unique():
        print(f"⚠ {var}: no files, skipping.")
        run_log.append({'var': var, 'status': 'no_files', 'shape': None})
        continue

    print(f"\n▶ Building {var} ...")
    try:
        stack = build_variable_stack(var, resampling=resampling)
        out_path = DATA_OUT / f'{var}_stack.nc'
        stack.to_netcdf(out_path, engine='netcdf4')
        print(f"  ✓ saved {out_path.name}  shape={stack.shape}")
        run_log.append({
            'var': var,
            'status': 'ok',
            'shape': str(stack.shape),
            'file': out_path.name,
        })
        del stack
    except Exception as e:
        print(f"  ✗ FAILED: {e}")
        traceback.print_exc()
        run_log.append({'var': var, 'status': 'fail', 'error': str(e)})

log_df = pd.DataFrame(run_log)
log_df.to_csv(LOG_DIR / 'build_log.csv', index=False)
print("\nBuild summary:")
print(log_df.to_string(index=False))

if len(load_errors) > 0:
    pd.DataFrame(load_errors, columns=['file', 'error']).to_csv(
        LOG_DIR / 'load_errors.csv', index=False
    )
    print(f"\n⚠ {len(load_errors)} files failed to load — see load_errors.csv")

In [ ]:
# =============================================================================
# Apply NOBS QA to NDVI and LST.
# Uses open_dataset + variable selection because the files contain more than
# one data variable (data + year/season coords).
# =============================================================================

def qa_mask(target_var, nobs_var):
    tgt_ds  = xr.open_dataset(DATA_OUT / f'{target_var}_stack.nc')
    nobs_ds = xr.open_dataset(DATA_OUT / f'{nobs_var}_stack.nc')

    tgt  = tgt_ds[target_var]
    nobs = nobs_ds[nobs_var]

    nobs = nobs.reindex_like(tgt)

    masked = tgt.where(nobs >= NOBS_MIN)
    masked.attrs = dict(tgt.attrs)
    masked.attrs['qa'] = f'NOBS >= {NOBS_MIN}'

    out_path = DATA_OUT / f'{target_var}_stack_qa.nc'
    masked.to_netcdf(out_path, engine='netcdf4')
    print(f"✓ {target_var}: QA applied → {out_path.name}")
    return masked

ndvi = qa_mask('NDVI', 'NOBS_NDVI')
lst  = qa_mask('LST',  'NOBS_LST')

In [ ]:
# =============================================================================
# SRTM is static — no time dimension.
# =============================================================================

srtm_path = DATA_IN / 'SRTM_ELEVATION.tif'
if not srtm_path.exists():
    # try alternative names
    candidates = list(DATA_IN.glob('SRTM*.tif'))
    if candidates:
        srtm_path = candidates[0]

print(f"SRTM file: {srtm_path.name}")

srtm = rxr.open_rasterio(srtm_path, masked=True).squeeze()
if srtm.rio.crs is None:
    srtm = srtm.rio.write_crs(CRS, inplace=False)

if srtm.shape != master_shape or str(srtm.rio.crs) != CRS:
    srtm = srtm.rio.reproject(
        CRS,
        shape=master_shape,
        transform=master_transform,
        resampling=Resampling.bilinear,
    )

srtm = srtm.where(srtm != NODATA).astype('float32')
srtm = srtm.rio.write_crs(CRS, inplace=False)
srtm.to_netcdf(DATA_OUT / 'SRTM_stack.nc', engine='netcdf4')
print(f"✓ SRTM saved. range = {float(srtm.min()):.0f}–{float(srtm.max()):.0f} m")

In [ ]:
# =============================================================================
# Summary of everything produced.
# Uses open_dataset + variable selection because saved stacks contain
# more than one data variable (data + year/season coords).
# =============================================================================

summary = []
for var in DATASETS.keys():
    qa  = DATA_OUT / f'{var}_stack_qa.nc'
    raw = DATA_OUT / f'{var}_stack.nc'
    path = qa if qa.exists() else raw
    if not path.exists():
        continue

    try:
        ds = xr.open_dataset(path)

        # Pick the variable matching this key. QA files keep the same
        # data-variable name as the raw stack (e.g. 'NDVI').
        if var in ds.data_vars:
            var_key = var
        else:
            # Fallback: match by prefix
            candidates = [k for k in ds.data_vars
                          if k == var or k.split('_')[0] == var.split('_')[0]]
            if not candidates:
                print(f"⚠ {var}: no matching data variable in {path.name}, "
                      f"found {list(ds.data_vars)}")
                ds.close()
                continue
            var_key = candidates[0]

        da = ds[var_key]
        vals = da.values
        finite = np.isfinite(vals)

        summary.append({
            'variable': var,
            'shape':    str(da.shape),
            'valid_%':  round(100 * finite.sum() / finite.size, 1),
            'min':      float(np.nanmin(vals)) if finite.any() else np.nan,
            'max':      float(np.nanmax(vals)) if finite.any() else np.nan,
            'file':     path.name,
        })
        ds.close()

    except Exception as e:
        print(f"✗ {var}: {e}")
        summary.append({
            'variable': var,
            'shape':    None,
            'valid_%':  None,
            'min':      None,
            'max':      None,
            'file':     path.name,
        })

summary_df = pd.DataFrame(summary)
print(summary_df.to_string(index=False))
summary_df.to_csv(LOG_DIR / 'preprocessing_summary.csv', index=False)

In [ ]:
# =============================================================================
# Quick visual check: NDVI for JF 2010 and LST for JJAS 2010.
# =============================================================================

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ndvi = xr.open_dataarray(DRIVE_OUT / 'NDVI_stack_qa.nc')
lst  = xr.open_dataarray(DRIVE_OUT / 'LST_stack_qa.nc')

sel_ndvi = ndvi.sel(time=('2010', 'JF')).squeeze()
sel_lst  = lst.sel(time=('2010', 'JJAS')).squeeze()

axes[0].imshow(sel_ndvi, cmap='YlGn', vmin=0, vmax=0.9)
axes[0].set_title('NDVI — JF 2010')
axes[0].axis('off')

axes[1].imshow(sel_lst, cmap='inferno')
axes[1].set_title('LST (K) — JJAS 2010')
axes[1].axis('off')

plt.tight_layout()
plt.savefig(DRIVE_OUT / 'preview.png', dpi=120)
plt.show()

In [ ]:
# =============================================================================
# FIRMS fix — unmask NaN-in-AOI to 0.
#
# FIRMS is a COUNT variable. Zero fire detections is a real value, not
# "missing data". The GEE export used unmask(-9999), which Cell 4 then
# converted to NaN — losing all zero-detection pixels. This cell restores
# those zeros within the AOI for years >= 2001.
#
# What stays NaN:
#   - Pixels outside the AOI footprint
#   - Seasons before FIRMS coverage (i.e. before 2001)
# =============================================================================

firms_path = DATA_OUT / 'FIRE_DAYS_stack.nc'
ds_firms = xr.open_dataset(firms_path)
firms = ds_firms['FIRE_DAYS']

# --- Build the "inside AOI" mask from a Landsat-derived variable ---
# NDVI_stack_qa has NaN outside the AOI (because of clip in GEE) and NaN
# inside the AOI where cloud/QA killed the pixel. We want the AOI footprint,
# so use the raw NOBS_NDVI stack: NaN there = outside AOI or no scenes.
nobs_ds = xr.open_dataset(DATA_OUT / 'NOBS_NDVI_stack.nc')
nobs_ndvi = nobs_ds['NOBS_NDVI']

# A pixel is "in AOI" for a given time if NOBS_NDVI is not NaN.
# If NOBS is NaN because there were no Landsat scenes that season, we
# want to still treat it as in-AOI. So use the *spatial* union across time:
aoi_mask_2d = nobs_ndvi.notnull().any(dim='time')   # (y, x) boolean

print(f"AOI footprint (from NOBS_NDVI union): "
      f"{int(aoi_mask_2d.sum()):,} pixels of "
      f"{aoi_mask_2d.size:,} ({100*float(aoi_mask_2d.sum())/aoi_mask_2d.size:.1f}%)")

# --- Build the "year >= 2001" mask ---
years = ds_firms['year'].values
year_mask = years >= 2001          # shape (time,)
print(f"Seasons with FIRMS coverage: {int(year_mask.sum())} of {len(years)}")

# --- Apply the fix ---
# Where:
#   - AOI mask is True (spatially inside AOI)
#   - year >= 2001
#   - FIRMS is NaN
# → set to 0. Everywhere else, keep original value (including NaN outside AOI).

aoi_3d   = aoi_mask_2d.broadcast_like(firms.isel(time=0)).expand_dims(time=firms.time)
year_3d  = xr.DataArray(year_mask, dims='time', coords={'time': firms.time})

fill_here = aoi_3d & year_3d & firms.isnull()
firms_fixed = firms.where(~fill_here, 0.0)

# --- Preserve coords / attrs ---
firms_fixed.name = 'FIRE_DAYS'
firms_fixed.attrs = dict(firms.attrs)
firms_fixed.attrs['note'] = 'Zeros restored within AOI for years >= 2001'
firms_fixed = firms_fixed.rio.write_crs(CRS, inplace=False)

# --- Save ---
out_path = DATA_OUT / 'FIRE_DAYS_stack_fixed.nc'
firms_fixed.to_netcdf(out_path, engine='netcdf4')
print(f"✓ FIRMS fixed → {out_path.name}")

# --- Quick sanity check ---
before_valid = float(firms.notnull().sum()) / firms.size
after_valid  = float(firms_fixed.notnull().sum()) / firms_fixed.size
print(f"  Valid before: {100*before_valid:.2f}%")
print(f"  Valid after : {100*after_valid:.2f}%")

# Value distribution after fix
import numpy as np
vals = firms_fixed.values
finite = np.isfinite(vals)
zeros = (vals == 0) & finite
nonzero = (vals > 0) & finite
print(f"  Zeros       : {int(zeros.sum()):,}")
print(f"  Nonzero     : {int(nonzero.sum()):,}")
print(f"  Still NaN   : {int((~finite).sum()):,}")

ds_firms.close()
nobs_ds.close()